# Vector autoregression and impulse responses: lab notebook

**Statistical Modelling and Forecasting, ARIMA and vector autoregression**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.api import VAR
from statsmodels.tsa.vector_ar.var_model import VARResults

### 1. Growth of consumption, investment and GDP (quarterly, log differences in percent)

In [ ]:
# 1. Growth of consumption, investment and GDP (quarterly, log differences in percent)
d = sm.datasets.macrodata.load_pandas().data
d.index = pd.period_range("1959Q1", periods=len(d), freq="Q")
X = 100 * np.log(d[["realgdp", "realcons", "realinv"]]).diff().dropna()
X.columns = ["gdp", "cons", "inv"]
print(X.describe().loc[["mean", "std"]].round(3).to_string())

### 2. Lag selection by information criteria

In [ ]:
# 2. Lag selection by information criteria
model = VAR(X)
sel = model.select_order(maxlags=6)
print(sel.summary())
res = model.fit(2)
print("stable (all roots outside the unit circle):", res.is_stable())
print("largest eigenvalue modulus of the companion matrix:", round(float(1 / np.min(np.abs(res.roots))), 3))
print("coefficient matrix A1:")
print(pd.DataFrame(res.coefs[0], index=X.columns, columns=X.columns).round(3).to_string())

### 3. Granger causality tests

In [ ]:
# 3. Granger causality tests
for cause in ["cons", "inv"]:
    t = res.test_causality("gdp", [cause], kind="f")
    print(f"{cause} -> gdp: F = {t.test_statistic:.2f}, p = {t.pvalue:.3f}")

### 4. Impulse responses (Cholesky, order gdp, cons, inv) and a check of the ordering

In [ ]:
# 4. Impulse responses (Cholesky, order gdp, cons, inv) and a check of the ordering
irf = res.irf(8)
print("orthogonal response of gdp to a one-sd shock in cons (h = 0..4):", np.round(irf.orth_irfs[:5, 0, 1], 3))
rev = VAR(X[["inv", "cons", "gdp"]]).fit(2).irf(8)
print("same shock with the reversed ordering:                          ", np.round(rev.orth_irfs[:5, 2, 1], 3))

### 5. Forecast and variance decomposition

In [ ]:
# 5. Forecast and variance decomposition
fc = res.forecast(X.values[-2:], steps=4)
print("4-step forecasts of growth (gdp, cons, inv):")
print(pd.DataFrame(fc, columns=X.columns).round(2).to_string())
fevd = res.fevd(8)
print("share of the 8-step gdp forecast variance due to each shock:", np.round(fevd.decomp[0, -1], 3))

### 6. The numerical problem

In [ ]:
# 6. The numerical problem
A = np.array([[0.6, 0.2], [0.4, 0.5]])
y = np.array([10.0, 12.0])
print("forecast T+1:", A @ y, "| T+2:", A @ A @ y, "| response of y2 after 2 steps to a y1 shock:", (A @ A)[1, 0])
print("eigenvalues:", np.round(np.linalg.eigvals(A), 3), "| det:", np.linalg.det(A))

## Expected output

Running all cells above should print:

```
        gdp   cons    inv
mean  0.776  0.837  0.814
std   0.880  0.694  4.685
 VAR Order Selection (* highlights the minimums) 
=================================================
      AIC         BIC         FPE         HQIC   
-------------------------------------------------
0    -0.05234   -0.002166      0.9490    -0.03203
1    -0.3822*    -0.1815*     0.6824*    -0.3009*
2     -0.3633    -0.01205      0.6955     -0.2211
3     -0.3642      0.1376      0.6950     -0.1610
4     -0.3721      0.2802      0.6897     -0.1080
5     -0.3475      0.4553      0.7072    -0.02248
6     -0.3132      0.6401      0.7324     0.07271
-------------------------------------------------
stable (all roots outside the unit circle): True
largest eigenvalue modulus of the companion matrix: 0.614
coefficient matrix A1:
        gdp   cons    inv
gdp  -0.279  0.675  0.033
cons -0.100  0.269  0.026
inv  -1.971  4.414  0.225
cons -> gdp: F = 16.97, p = 0.000
inv -> gdp: F = 0.81, p = 0.445
orthogonal response of gdp to a one-sd shock in cons (h = 0..4): [0.    0.299 0.211 0.076 0.069]
same shock with the reversed ordering:                           [0.384 0.33  0.261 0.099 0.085]
4-step forecasts of growth (gdp, cons, inv):
    gdp  cons   inv
0  0.50  0.54  0.51
1  0.59  0.78 -0.30
2  0.66  0.76  0.39
3  0.73  0.80  0.66
share of the 8-step gdp forecast variance due to each shock: [0.801 0.187 0.012]
forecast T+1: [ 8.4 10. ] | T+2: [7.04 8.36] | response of y2 after 2 steps to a y1 shock: 0.44
eigenvalues: [0.837+0.j 0.263+0.j] | det: 0.22
```

## Practice exercises

Try these in the cells below.

1. Set a₁₂ = 0 and a₂₁ = 0.4. Variable 1 does not respond to 2. Run the Granger test in both directions.
2. Make the diagonal 0.9 and 0.9 and the off-diagonals 0.3 and 0.3. The system is close to unstable. Read the IRF half-life.
3. Switch the Cholesky ordering with the correlation set to 0.6. Compare the first-period IRF.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
